[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HumbertoDiego/AjustamentoBasicoIME/blob/main/05_mmq_weighted.ipynb)

# Ajustamento Básico - MMQ ponderado
**Maj Diego - 2° Semestre / 2026**

**Objetivos:**

1. Formular e aplicar pesos às observações no MMQ
2. Formular o tratamento estatístico para análise do ajustamento
3. Interpretar o tratamento estatístico para análise do ajustamento

**Referência:**

Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley. $\rightarrow$ **Cap 5, 10, 13 e 25**

## O Problema

Até aqui, no ajustamento MMQ, a precisão do ajustamento/modelo é apenas informada ao final. Por exemplo, o caso abaixo:

$$
\begin{cases}
x = 10 \pm 0.1 \\
x = 14 \pm 5
\end{cases}
$$

$$x_a = 12 \ \ e \ \  \sigma_{x_a} = \sqrt{0.01^2 + 5^2} = 5.00000999999 $$

Como levar em consideração a precisão das observações durante o ajustamento?


## 1. Formular e aplicar pesos às observações no MMQ

### **1.1 Requisitos de uma matriz de pesos ($P$)**

- O peso de uma observação é uma medida do <b style="color:#2ECC40">valor relativo dessa observação em comparação com outras observações</b>.
- Os pesos são usados para <b style="color:#2ECC40">controlar as magnitudes das correções</b> aplicadas às observações em um ajustamento. 
- Quanto mais precisa for uma observação, maior deve ser o seu peso; em outras palavras, quanto menor a variância, maior o peso. 
- Intuitivamente, os pesos são <b style="color:#2ECC40">inversamente proporcionais às variâncias</b>.

> Nota: Consideraremos daqui para frente, observações não correlacionadas (independentes), assim $\Sigma$ e $P$ serão sempre matrizes diagonais.

### **1.2 Uma possível matriz de pesos ($P$)**

Da discussão dos requisitos para a matriz de pesos, podemos definir $P$ relacionada com a inversa da matriz de covariância, $\Sigma$. Como os pesos são relativos, variâncias e covariâncias podem ser reescalonadas tais que:

$$w_{i} = \frac{\sigma^2_0}{\sigma_{i}}  \rightarrow P = \sigma^2_0 \Sigma^{-1}$$

onde $w_{i}$ é o peso da observação $i$, $\sigma_{i}$ a covariância da observação $i$, e $\sigma^2_0$ a variância de referência, um valor que pode ser usado para escala. 

### **1.3 MMQ ponderado**

Para encaixar essa matriz de pesos dentro do MMQ. Seja $AX = L$ um sistema linear:

| $\begin{matrix}\text{Não ponderado} & & \end{matrix}$ | $\begin{matrix}\text{Ponderado} & & \end{matrix}$  |
|- | - |
|$\small AX_a = L_a + V$ | $\small PAX_a = PL_a + PV$| 
|$\small \sum v_i^2 = V^TV = min$ | $\small \sum w_iv_i^2 = V^TPV = min$| 
|$\small X_a​=(A^TA)^{−1}A^TL$ | $\small X_a​=(A^TPA)^{−1}A^TPL$|

E sendo $F(X) = L$ um sistema não linear e $J$ a jacobiana de $F$ :

| $\begin{matrix}\text{Não ponderado} & & \end{matrix}$ | $\begin{matrix}\text{Ponderado} & & \end{matrix}$  |
|- | - |
|$\small F(X_0 + \Delta X) \approx F(X_0) + J \Delta X = L_b + V$ | $\small PF(X_0 + \Delta X) \approx PF(X_0) + PJ \Delta X = PL_b + PV$| 
|$\small \sum v_i^2 = V^TV = min$ | $\small \sum w_iv_i^2 = V^TPV = min$| 
|$\small \Delta X​=-(J^TJ)^{−1}J^TL$ | $\small  \Delta X​=-(J^TPJ)^{−1}J^TPL$|


### **1.4 Propagação do erros no MMQ ponderado**

A solução MMQ é:

$$
X_a = (A^T P A)^{-1}A^T P L
$$

Pela lei de propagação das variâncias:

$$
\Sigma_{X_a} = ((A^T P A)^{-1}A^T P ) \Sigma_L ((A^T P A)^{-1}A^T P )^T
$$

Mas, $\Sigma_L =  \sigma_0^2P^{-1}$, então essa expressão se simplifica para:

$$
\boxed{
\Sigma_{X_a} = \sigma_0^2 (A^T P A)^{-1}
}
$$



**Exercício 01 (resolvido)** (Exercício 13.15 do livro GHILANI &amp; WOLF) O quadro e o esquema que se seguem resumem um nivelamento geométrico com referências de nível os pontos A e B, de altitude 263,453m e 294,837m respectivamente; as setas indicam o sentido da visada. Ajuste os desníveis pelo método dos mínimos quadrados paramétrico <b>considerando a precisão das observações indicada no quadro.</b>

<center > <img src="media/imgs/WolfProblem13-15esquema.png" style="height:300px"> <img src="media/imgs/WolfProblem13-15quadro.png" style="height:300px"></center>

(a) **elevação mais provável** para cada uma das estações **V, W, X, Y e Z**?

(b) **erro estimado** em cada elevação? 

(c) **observações ajustadas**, seus **resíduos** e **erro total**?

(d) **diferença de elevação** do marco de referência **BM A** até a estação **Z** e seu **erro estimado**? 

In [86]:
import numpy as np

# Sistema linear {
# 263.453 V  25.102  +- 0.018
# 294.837 V  -6.287  +- 0.019
# V       X  10.987  +- 0.016
# V       Y  24.606  +- 0.021
# 294.837 Y  17.993  +- 0.017
# 263.453 X  36.085  +- 0.021
# Y       X  -13.295 +- 0.018
# Y       Z  -20.732 +- 0.022
# W       Z  18.455  +- 0.022
# V       W  -14.896 +- 0.021
# 263.453 W  10.218  +- 0.017
# 294.837 X  4.693   +- 0.020
# W       X  25.883  +- 0.018
# X       Z  -7.456  +- 0.020

# Matriz design
#              V   X   Y   Z   W 
A = np.array([[1,  0,  0,  0,  0], 
              [1,  0,  0,  0,  0], 
              [-1, 1,  0,  0,  0], 
              [-1, 0,  1,  0,  0], 
              [0,  0,  1,  0,  0], 
              [0,  1,  0,  0,  0], 
              [0,  1, -1,  0,  0], 
              [0,  0, -1,  1,  0], 
              [0,  0,  0,  1, -1], 
              [-1, 0,  0,  0,  1], 
              [0,  0,  0,  0,  1], 
              [0,  1,  0,  0,  0], 
              [0,  1,  0,  0, -1], 
              [0,  -1, 0,  1,  0]])
# Vetor de incógnitas
# X = [[V], 
#      [X], 
#      [Y], 
#      [Z], 
#      [W]]
# Matriz das observações
L0 = np.array([[263.453], 
              [294.837],
              [0],
              [0],
              [294.837],
              [263.453],
              [0],
              [0],
              [0],
              [0],
              [263.453],
              [294.837],
              [0],
              [0]])
L = np.array([[25.102], 
              [-6.287], 
              [10.987], 
              [24.606], 
              [17.993], 
              [36.085], 
              [-13.295], 
              [-20.732], 
              [18.455], 
              [-14.896], 
              [10.218], 
              [4.693], 
              [25.883], 
              [-7.456]])
L = L + L0
# Matriz de pesos inversamente proporcional às variâncias
dp = np.array([0.018, 
               0.019, 
               0.016, 
               0.021, 
               0.017, 
               0.021, 
               0.018, 
               0.022, 
               0.022, 
               0.021, 
               0.017, 
               0.020, 
               0.018, 
               0.020]) # em m
var = dp**2 # em m^2
SigmaL =  np.diag(var) # em [m^2]
# # SigmaL = SigmaL * 1000000 # em m^2 # na verdade, qualquer fator de escala não importa
sigmapriori = 1 # fator de escala
P = sigmapriori * np.linalg.inv(SigmaL)
G = np.linalg.inv(A.T @ P @ A) @ A.T @ P
# SigmaXa = G @ SigmaL @ G.T
SigmaXa = sigmapriori * np.linalg.inv(A.T @ P @ A)
dpXa = np.diagonal(SigmaXa)**0.5
# ## MMQ
Xa = G @ L
V = A @ Xa - L
La = L + V

# Erro total 
gl = A.shape[0] - A.shape[1]
errototal = ( V.T @ P @ V).item() 
sigmaposteriori = ( V.T @ P @ V).item() / gl

print("a) X ajustado Xa=[V   X   Y   Z   W ]:")
print(Xa.reshape(-1))
print("\nb) Erros em Xa" )
print(dpXa)
print("\nc) L ajustado:")
print(La.reshape(-1))
print("Resíduos:")
print(V.reshape(-1))
print("Erro total ajustamento:")
print(errototal**0.5)
print("\nd) BM A - Z:", 263.453 - Xa.reshape(-1)[-2] )
# Lei de propagação das variâncias:
# f =  Cte + [-1][Z]-> Sigmaf = [-1][SigmaZ][-1]^T = SigmaZ
print("Erro do desnível (BM A - Z):", dpXa[3] )

a) X ajustado Xa=[V   X   Y   Z   W ]:
[288.5125766  299.54520128 312.89714479 292.11963598 273.65595316]

b) Erros em Xa
[0.00964385 0.00962074 0.01122239 0.01472935 0.01122239]

c) L ajustado:
[288.5125766  288.5125766   11.03262467  24.38456819 312.89714479
 299.54520128 -13.35194351 -20.77750881  18.46368282 -14.85662345
 273.65595316 299.54520128  25.88924812  -7.4255653 ]
Resíduos:
[-0.0424234  -0.0374234   0.04562467 -0.22143181  0.06714479  0.00720128
 -0.05694351 -0.04550881  0.00868282  0.03937655 -0.01504684  0.01520128
  0.00624812  0.0304347 ]
Erro total ajustamento:
12.892747648022318

d) BM A - Z: -28.666635978821205
Erro do desnível (BM A - Z): 0.014729350144162844


## 2. Formular o tratamento estatístico para análise do ajustamento

### **2.1 Variância a priori e a posteriori:**

No contexto do ajustamento, vamos chamar o nosso "fator de escala" $\sigma_0^2$ de variância a *priori*. Ela será importante ao ser comparada com a variância após os cálculos do ajustamento, ou seja uma variância a *posteriori* assim definida:

$$\hat{\sigma}_0^2 = \frac{V^TPV}{m-n}$$

onde $m$ é o número de observações/equações, $n$ é a quantidade de parâmetros/incógnitas, $m-n$ é o grau de liberdade $(gl)$ do sistema linear em questão.

### **2.1 Variância a priori e a posteriori:**

Ao compararmos podemos encontrar três situações:
- Se $\hat{\sigma}_0^2 > \sigma_0^2$, então a qualidade das observações é **pior** do que a suposta
- Se $\hat{\sigma}_0^2 = \sigma_0^2$, então a qualidade das observações é conforme a suposta
- Se $\hat{\sigma}_0^2 < \sigma_0^2$, então a qualidade das observações é **melhor** do que a suposta $\rightarrow$ Qual o problema nisso?

### **2.2 Teste de hipótese para variância**

<p style="margin:0 0 0 0">Esse teste visa <b style="color:#2ECC40">verificar se uma medida está de acordo com sua precisão publicada</b>, para isso a distribuição Qui-quadrado (χ²) é usada para comparação entre variâncias da população/publicada/esperada e da amostra/calculada, consistindo em:</p>

<center><img src="media/imgs/testequiquadrado.png" style="margin:0 0 0 0"></center>

<p style="margin:0 0 0 0">onde α é o <b>nível de significância</b>, ou seja, a probabilidade de rejeitar $H_0$ se ela é verdadeira, seu complemento para unidade é o <b>nível de confiança</b> (1-α), ou seja, a probabilidade de aceitar $H_0$ se ela é verdadeira</p>

**Distribuição Qui-Quadrada em função dos graus de liberdade e significância:**

<center><img src="media/imgs/bilateral.png"></center>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from ipywidgets import AppLayout, FloatSlider, IntSlider, VBox, HBox, Label, Output
from IPython.display import display, clear_output

# Configurações de estilo do Matplotlib
# plt.style.use('ggplot')

def criar_simulador_qui_quadrado():
    # --- Componentes da Interface ---
    df_slider = IntSlider(value=10, min=2, max=100, step=1, description='Graus (gl):')
    alpha_slider = FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description='Signif. (α):')
    calc_slider = FloatSlider(value=12.5, min=0, max=80, step=0.1, description='χ² Calc:')
    
    out = Output()

    def atualizar_grafico(change=None):
        df = df_slider.value
        alpha = alpha_slider.value
        x_calc = calc_slider.value
        
        # 1. Cálculos Estatísticos
        # No bilateral, dividimos alpha nas duas extremidades
        crit_inf = stats.chi2.ppf(alpha/2, df)
        crit_sup = stats.chi2.ppf(1 - alpha/2, df)
        
        # 2. Definição do eixo X
        x_max = max(stats.chi2.ppf(0.999, df), x_calc + 5)
        x = np.linspace(0, x_max, 500)
        y = stats.chi2.pdf(x, df)
        
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 6))
            
            # Curva principal
            ax.plot(x, y, '-', lw=2, label=fr'Distribuição $\chi^2$ ({df} gl)')
            
            # Áreas de Rejeição (Caudas)
            x_left = np.linspace(0, crit_inf, 100)
            ax.fill_between(x_left, stats.chi2.pdf(x_left, df), color='red', alpha=0.3, label='Região Crítica')
            
            x_right = np.linspace(crit_sup, x_max, 100)
            ax.fill_between(x_right, stats.chi2.pdf(x_right, df), color='red', alpha=0.3)
            
            # Marcador do Valor Calculado
            ax.axvline(x_calc, color='green', linestyle='--', lw=2)
            ax.scatter(x_calc, stats.chi2.pdf(x_calc, df), color='green', s=100, zorder=5, label=f'Calculado: {x_calc:.2f}')
            
            # Anotações dos Valores Críticos
            ax.annotate(fr'$\chi^2({alpha/2},{df})\approx$ {crit_inf:.2f}', xy=(crit_inf, 0), xytext=(crit_inf, -0.01),
                        arrowprops=dict(arrowstyle='-|>', color='red'), ha='center', color='red', annotation_clip=False)
            ax.annotate(fr'$\chi^2({1 - alpha/2},{df})\approx$ {crit_sup:.2f}', xy=(crit_sup, 0), xytext=(crit_sup, -0.01),
                        arrowprops=dict(arrowstyle='-|>', color='red'), ha='center', color='red', annotation_clip=False)
            
            # Centro das regiões (aproximado)
            x_left_center = crit_inf / 2
            x_mid_center = (crit_inf + crit_sup) / 2
            x_right_center = crit_sup + (x.max() - crit_sup)/2

            ax.annotate('2.5%',
                        xy=(x_left_center, stats.chi2.pdf(x_left_center, df)),
                        # xytext=(x_left_center, stats.chi2.pdf(x_left_center, df)),
                        arrowprops=None, ha='center', color='red', fontsize=14, fontweight='bold')

            ax.annotate('95%',
                        xy=(x_mid_center, stats.chi2.pdf(x_mid_center, df)/2),
                        # xytext=(x_mid_center, stats.chi2.pdf(x_mid_center, df)),
                        arrowprops=None, ha='center', color='blue', fontsize=14, fontweight='bold')

            ax.annotate('2.5%',
                        xy=(x_right_center, stats.chi2.pdf(x_right_center, df)),
                        # xytext=(x_right_center, stats.chi2.pdf(x_right_center, df)),
                        arrowprops=None, ha='center', color='red', fontsize=14, fontweight='bold')
            

            
            # Lógica de Decisão
            decisao = "NÃO REJEITA H₀"
            cor_decisao = "darkgreen"
            if x_calc < crit_inf or x_calc > crit_sup:
                decisao = "REJEITA H₀"
                cor_decisao = "darkred"

            ax.set_title(r"Teste $\chi^2$ bilateral ($H_1: \hat{\sigma}^2 \neq \sigma_0^2$)", fontsize=14)
            ax.text(x_max*0.8, max(y)*0.7, decisao, fontsize=13, fontweight='bold', color=cor_decisao, 
                    bbox=dict(facecolor='white', alpha=0.8))
            
            # ax.set_xlim(0, 12)
            ax.set_ylim(0, max(y)*1.05)
            ax.legend()
            fig.canvas.header_visible = False

            # Linhas verticais
            ax.axvline(crit_inf, linestyle='--')
            ax.axvline(crit_sup, linestyle='--')
            
            plt.show()

    # Observadores para tornar o gráfico dinâmico
    df_slider.observe(atualizar_grafico, names='value')
    alpha_slider.observe(atualizar_grafico, names='value')
    calc_slider.observe(atualizar_grafico, names='value')
    
    # Layout usando AppLayout
    controles = VBox([
        Label(value="Configurações do Teste"),
        df_slider, alpha_slider, calc_slider
    ])

    layout = AppLayout(
        header=None,
        left_sidebar=controles,
        center=out,
        right_sidebar=None,
        footer=None,
        pane_widths=['350px', '1fr', 0],
        grid_gap='20px'
    )
    
    display(layout)
    atualizar_grafico() # Chamada inicial

# Executar o simulador
criar_simulador_qui_quadrado()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import ipywidgets as widgets
from ipywidgets import AppLayout, FloatSlider, IntSlider, VBox, HBox, Label, Output
from IPython.display import display, clear_output

# Configurações de estilo do Matplotlib
# plt.style.use('ggplot')

def criar_simulador_qui_quadrado():
    # --- Componentes da Interface ---
    df_slider = IntSlider(value=10, min=2, max=100, step=1, description='Graus (gl):')
    alpha_slider = FloatSlider(value=0.05, min=0.01, max=0.20, step=0.01, description='Signif. (α):')
    calc_slider = FloatSlider(value=12.5, min=0, max=80, step=0.1, description='χ² Calc:')
    
    out = Output()

    def atualizar_grafico(change=None):
        df = df_slider.value
        alpha = alpha_slider.value
        x_calc = calc_slider.value
        
        # 1. Cálculos Estatísticos
        # No bilateral, dividimos alpha nas duas extremidades
        # crit_inf = stats.chi2.ppf(alpha/2, df)
        crit_sup = stats.chi2.ppf(1 - alpha, df)
        
        # 2. Definição do eixo X
        x_max = max(stats.chi2.ppf(0.999, df), x_calc + 5)
        x = np.linspace(0, x_max, 500)
        y = stats.chi2.pdf(x, df)
        
        with out:
            clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(10, 6))
            
            # Curva principal
            ax.plot(x, y, '-', lw=2, label=fr'Distribuição $\chi^2$ ({df} gl)')
            
            # Áreas de Rejeição (Caudas)
            # x_left = np.linspace(0, crit_inf, 100)
            # ax.fill_between(x_left, stats.chi2.pdf(x_left, df), color='red', alpha=0.3, label='Região Crítica')
            
            x_right = np.linspace(crit_sup, x_max, 100)
            ax.fill_between(x_right, stats.chi2.pdf(x_right, df), color='red', alpha=0.3)
            
            # Marcador do Valor Calculado
            ax.axvline(x_calc, color='green', linestyle='--', lw=2)
            ax.scatter(x_calc, stats.chi2.pdf(x_calc, df), color='green', s=100, zorder=5, label=f'Calculado: {x_calc:.2f}')
            
            # Anotações dos Valores Críticos
            # ax.annotate(fr'$\chi^2({alpha/2},{df})\approx$ {crit_inf:.2f}', xy=(crit_inf, 0), xytext=(crit_inf, -0.01),
            #             arrowprops=dict(arrowstyle='-|>', color='red'), ha='center', color='red', annotation_clip=False)
            ax.annotate(fr'$\chi^2({1 - alpha},{df})\approx$ {crit_sup:.2f}', xy=(crit_sup, 0), xytext=(crit_sup, -0.01),
                        arrowprops=dict(arrowstyle='-|>', color='red'), ha='center', color='red', annotation_clip=False)
            
            # Centro das regiões (aproximado)
            # x_left_center = crit_inf / 2
            # x_mid_center = (crit_inf + crit_sup) / 2
            x_right_center = crit_sup + (x.max() - crit_sup)/2

            # ax.annotate('2.5%',
            #             xy=(x_left_center, stats.chi2.pdf(x_left_center, df)),
            #             # xytext=(x_left_center, stats.chi2.pdf(x_left_center, df)),
            #             arrowprops=None, ha='center', color='red', fontsize=14, fontweight='bold')

            # ax.annotate('95%',
            #             xy=(x_mid_center, stats.chi2.pdf(x_mid_center, df)/2),
            #             # xytext=(x_mid_center, stats.chi2.pdf(x_mid_center, df)),
            #             arrowprops=None, ha='center', color='blue', fontsize=14, fontweight='bold')

            ax.annotate('5%',
                        xy=(x_right_center, stats.chi2.pdf(x_right_center, df)),
                        # xytext=(x_right_center, stats.chi2.pdf(x_right_center, df)),
                        arrowprops=None, ha='center', color='red', fontsize=14, fontweight='bold')
            

            
            # Lógica de Decisão
            decisao = "NÃO REJEITA H₀"
            cor_decisao = "darkgreen"
            if  x_calc > crit_sup:
                decisao = "REJEITA H₀"
                cor_decisao = "darkred"

            ax.set_title(r"Teste $\chi^2$ unilateral ($H_1: \hat{\sigma}^2 > \sigma_0^2$)", fontsize=14)
            ax.text(x_max*0.8, max(y)*0.7, decisao, fontsize=13, fontweight='bold', color=cor_decisao, 
                    bbox=dict(facecolor='white', alpha=0.8))
            
            # ax.set_xlim(0, 12)
            ax.set_ylim(0, max(y)*1.05)
            ax.legend()
            fig.canvas.header_visible = False

            # Linhas verticais
            # ax.axvline(crit_inf, linestyle='--')
            ax.axvline(crit_sup, linestyle='--')
            
            plt.show()

    # Observadores para tornar o gráfico dinâmico
    df_slider.observe(atualizar_grafico, names='value')
    alpha_slider.observe(atualizar_grafico, names='value')
    calc_slider.observe(atualizar_grafico, names='value')
    
    # Layout usando AppLayout
    controles = VBox([
        Label(value="Configurações do Teste"),
        df_slider, alpha_slider, calc_slider
    ])

    layout = AppLayout(
        header=None,
        left_sidebar=controles,
        center=out,
        right_sidebar=None,
        footer=None,
        pane_widths=['350px', '1fr', 0],
        grid_gap='20px'
    )
    
    display(layout)
    atualizar_grafico() # Chamada inicial

# Executar o simulador
criar_simulador_qui_quadrado()

**Exercício 02 (resolvido)** (Exemplo 5.3 GHILANI &amp; WOLF) O proprietário de uma empresa de levantamentos topográficos deseja que todos os técnicos em agrimensura sejam capazes de realizar a leitura de um determinado instrumento com precisão de até **±1.5"**. Para testar esse valor, o proprietário pede ao chefe de campo mais experiente que realize um teste de leitura com o instrumento.

O chefe de campo lê o círculo **30 vezes** e obtém:

$$
s = \pm 0.9"
$$

Esse resultado sustenta o limite de **1.5"** ao nível de significância de **5%**?


**Solução exercício 02**
Neste caso, o proprietário deseja testar a hipótese de que a variância amostral calculada $(s^2)$ é igual à variância populacional $(\sigma^2)$, em vez de ser maior que a variância populacional. Ou seja, todos os desvios-padrão iguais ou menores que **1,5"** serão aceitos.

Assim, constrói-se um teste unilateral à direita, conforme segue. Observe que:

$$ \small gl = 30 - 1 = 29$$

A hipótese nula e a hipótese alternativa são :

$$ \small H_0: s^2 = \sigma^2 \text{ e } H_1: s^2 > \sigma^2 $$

A estatística de teste é:

$$ \small \chi^2 = \frac{gl . s^2}{\sigma^2} = \frac{(30-1)(0.9)^2}{(1.5)^2} = 10.44$$

A hipótese nula é rejeitada quando a estatística de teste calculada excede o valor tabelado. Ou seja, quando a seguinte condição for verdadeira:

$$
\small \chi^2 = 10.44 > \chi^2_{1-\alpha, gl} = \chi^2_{0.95;29} = 42,56 \leftarrow \text{scipy.stats.chi2.ppf(1 - alpha, gl)}
$$

logo falsa, a **hipótese nula não pode ser rejeitada.**

<!-- No entanto, simplesmente deixar de rejeitar a hipótese nula não significa que o valor de **±1,5"** seja válido. Este exemplo demonstra um problema comum em testes estatísticos: a interpretação incorreta dos resultados.

Uma amostra válida da população de todos os funcionários de levantamentos topográficos não pode ser obtida selecionando apenas um empregado. Além disso, o teste é falho porque cada funcionário pode ler o instrumento de forma diferente. Funcionários novos podem inicialmente apresentar leituras diferentes devido à falta de experiência com o instrumento.

Para considerar adequadamente essa falta de experiência, o empregador poderia testar uma amostra aleatória de candidatos durante a entrevista e repetir o teste após vários meses de trabalho. O proprietário poderia então verificar se existe correlação entre a satisfação da empresa com o funcionário e a habilidade inicial desse funcionário em ler o instrumento. No entanto, é improvável que alguma correlação significativa fosse encontrada.

Este é um exemplo de **uso inadequado da estatística**. Ele ilustra um ponto importante: a interpretação de testes estatísticos requer julgamento por parte da pessoa que realiza o teste. Deve-se sempre lembrar que, em um teste estatístico, o objetivo é **rejeitar ou não rejeitar a hipótese nula**, e não “aceitá-la” como verdadeira.

Além disso, um teste estatístico deve ser usado apenas quando for apropriado. -->

## 3. Interpretar o tratamento estatístico para análise do ajustamento

Vimos que a expressão para obtenção das variâncias dos parâmetros ajustados é

$$
\Sigma_{X_a} = \sigma_0^2 (A^T P A)^{-1}
$$

Se o teste $\chi^2$ <b style="color:#CC2E40">rejeitou a hipótese nula</b> $(H_0 \ :\ \sigma^2_0 = \hat{\sigma}^2_0)$, logo, mantém-se a variância a *priori* $(\sigma^2_0)$. Lembrando que $P =  \sigma_0^2\Sigma_L^{-1}$, pode-se ir além na simplificação:

$$
\Sigma_{X_a} = \sigma^2_0 (A^T \sigma_0^2\Sigma_L^{-1} A)^{-1} = (A^T \Sigma_L^{-1} A)^{-1}
$$

Se o teste $\chi^2$ <b style="color:#2ECC40">não rejeitou a hipótese nula</b> $(H_0 \ :\ \sigma^2_0 = \hat{\sigma}^2_0)$, logo, em termos estatísticos, é mais apropriado utilizar a variância da amostra, a calculada a partir dos dados, ou seja a variância a *posteriori* $(\hat{\sigma}^2_0)$. Assim obtemos uma nova expressão para as variâncias dos parâmetros ajustados:

$$
\Sigma_{X_a} = \hat{\sigma}^2_0 (A^T P A)^{-1}
$$

## Lista de exercícios complementares

**Exercício 03**: (Exercício 7.8.4 do livro Gemael) 

O quadro e o esquema que se seguem resumem um nivelamento geométrico que partiu da referência de nível A, de altitude nula; as setas indicam o sentido em que o terreno se eleva.

<img src="media/imgs/img13.jpeg" style="margin:auto" >

| LINHA | DESNÍVEL (m) | COMPRIMENTO (km) |
|-------|--------------|------------------|
| 1     | 6,16         | 4                |
| 2     | 12,57        | 2                |
| 3     | 6,41         | 2                |
| 4     | 1,09         | 4                |
| 5     | 11,58        | 2                |
| 6     | 5,07         | 4                |

a) Estimar as altitudes das estações B, C e D pelo MMQ. Adotar desvio padrão em Km igual a raiz do comprimento da linha em Km (não ligue para a incoerência da grandeza neste momento), ou seja, $\sigma_i = \sqrt{d_i}$, e ainda, pesos inversamente proporcionais às variâncias.

b) Tem sentido realizar o teste Qui-quadrado para aceitação destas observações dada essa adoção que $\sigma_i = \sqrt{d_i}$? 

c) Qual seria a melhor relação $\sigma_i = k \sqrt{d_i}$ para aceitação no teste?


**Exercício 04**: Volte ao **Exercício 01 (resolvido)** (Exercício 13.15 do livro GHILANI &amp; WOLF):

**a)** Faça um teste $(\chi^2)$ no Problema 13.15. Qual observação pode conter um **erro grosseiro**?

**b)** Repita o ajustamento **sem a observação 4**.

**Gabarito exercício 04 (comentado)**

<!-- print("\nErro total ajustamento:")
print(errototal**0.5)

from scipy import stats

alpha = 0.05
qui = ( V.T @ P @ V).item()

print("\nTeste unilateral:")
crit_sup = stats.chi2.ppf(1 - alpha, gl)
print("Chi > crit_sup?", f"{qui} > {crit_sup}?", (qui > crit_sup))

print("\nTeste bilateral:")
crit_inf = stats.chi2.ppf(alpha/2, gl)
crit_sup = stats.chi2.ppf(1 - alpha/2, gl)
print("Chi > crit_sup?", f"{qui} > {crit_sup}?", (qui > crit_sup))
print("Chi < crit_inf?", f"{qui} < {crit_inf}?", (qui < crit_sup))

import numpy as np

print(f"\n==== Removendo a observação com maior erro: V[{np.argmax(np.abs(V))}] = {V[np.argmax(np.abs(V))]} =====")

# Sistema linear {
# 263.453 V  25.102  +- 0.018
# 294.837 V  -6.287  +- 0.019
# V       X  10.987  +- 0.016
### V       Y  24.606  +- 0.021
# 294.837 Y  17.993  +- 0.017
# 263.453 X  36.085  +- 0.021
# Y       X  -13.295 +- 0.018
# Y       Z  -20.732 +- 0.022
# W       Z  18.455  +- 0.022
# V       W  -14.896 +- 0.021
# 263.453 W  10.218  +- 0.017
# 294.837 X  4.693   +- 0.020
# W       X  25.883  +- 0.018
# X       Z  -7.456  +- 0.020

# Matriz design
#              V   X   Y   Z   W 
A = np.array([[1,  0,  0,  0,  0], 
              [1,  0,  0,  0,  0], 
              [-1, 1,  0,  0,  0], 
              # [-1, 0,  1,  0,  0], 
              [0,  0,  1,  0,  0], 
              [0,  1,  0,  0,  0], 
              [0,  1, -1,  0,  0], 
              [0,  0, -1,  1,  0], 
              [0,  0,  0,  1, -1], 
              [-1, 0,  0,  0,  1], 
              [0,  0,  0,  0,  1], 
              [0,  1,  0,  0,  0], 
              [0,  1,  0,  0, -1], 
              [0,  -1, 0,  1,  0]])
# Vetor de incógnitas
# X = [[V], 
#      [X], 
#      [Y], 
#      [Z], 
#      [W]]
# Matriz das observações
L0 = np.array([[263.453], 
              [294.837],
              [0],
              # [0],
              [294.837],
              [263.453],
              [0],
              [0],
              [0],
              [0],
              [263.453],
              [294.837],
              [0],
              [0]])
L = np.array([[25.102], 
              [-6.287], 
              [10.987], 
              # [24.606], 
              [17.993], 
              [36.085], 
              [-13.295], 
              [-20.732], 
              [18.455], 
              [-14.896], 
              [10.218], 
              [4.693], 
              [25.883], 
              [-7.456]])
L = L + L0
# Matriz de pesos inversamente proporcional às variâncias
dp = np.array([0.018, 
               0.019, 
               0.016, 
              #  0.021, 
               0.017, 
               0.021, 
               0.018, 
               0.022, 
               0.022, 
               0.021, 
               0.017, 
               0.020, 
               0.018, 
               0.020]) # em m
var = dp**2 # em m^2
SigmaL =  np.diag(var) # em [m^2]
# # SigmaL = SigmaL * 1000000 # em m^2 # na verdade, qualquer fator de escala não importa
sigmapriori = 1 # fator de escala
P2 = sigmapriori * np.linalg.inv(SigmaL)
G = np.linalg.inv(A.T @ P2 @ A) @ A.T @ P2
# ## MMQ
Xa = G @ L
V2 = A @ Xa - L

# Erro total 
gl2 = A.shape[0] - A.shape[1]
errototal2 = ( V2.T @ P2 @ V2).item() 
sigmaposteriori = ( V2.T @ P2 @ V2).item() / gl2

print("\nErro total ajustamento:")
print(errototal2**0.5)

qui = ( V2.T @ P2 @ V2).item()
print("\nTeste unilateral:")
crit_sup = stats.chi2.ppf(1 - alpha, gl2)
print("Chi > crit_sup?", f"{qui} > {crit_sup}?", (qui > crit_sup))

print("\nTeste bilateral:")
crit_inf = stats.chi2.ppf(alpha/2, gl2)
crit_sup = stats.chi2.ppf(1 - alpha/2, gl2)
print("Chi > crit_sup?", f"{qui} > {crit_sup}?", (qui > crit_sup))
print("Chi < crit_inf?", f"{qui} < {crit_inf}?", (qui < crit_sup)) -->


## Lista de exercícios suplementares

Ghilani, C. D. (2017). *Adjustment computations: Spatial data analysis* (6th ed.). Wiley. 

Pág 254 (13. PRECISIONS OF INDIRECTLY DETERMINED QUANTITIES - Problems).

Pág 276 a 281 (14. ADJUSTMENT OF HORIZONTAL SURVEYS: TRILATERATION - Problems).

Pág 289, 303 a 312 (15. ADJUSTMENT OF HORIZONTAL SURVEYS: TRIANGULATION - Problems).